In [1]:
# ── CV Embedding -> ChromaDB (MiMo pipeline) ──────────────────────────
# Embedding: sentence-transformers lokal (MiMo tidak punya /embeddings endpoint)
# Jalankan SETELAH cv_semantic_chunking_mimo_v1 mengisi data/processed_mimo/*.json

from sentence_transformers import SentenceTransformer
import json
import chromadb
from pathlib import Path

EMBED_MODEL   = "BAAI/bge-m3"          # multilingual 1024-dim, download otomatis ~2GB
PROCESSED_DIR = Path("../../data/processed_mimo")
CHROMA_DIR    = Path("../../data/chroma")
COLLECTION    = "cv_chunks_mimo"

print("Loading embedding model (pertama kali download ~2GB, selanjutnya dari cache)...")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Model '{EMBED_MODEL}' siap | dimensi: {embedder.get_sentence_embedding_dimension()}")

Loading embedding model (pertama kali download ~2GB, selanjutnya dari cache)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model 'BAAI/bge-m3' siap | dimensi: 1024


/var/folders/2d/54t_6l6d5cl40cgyxw4l5_nm0000gn/T/ipykernel_11600/3589128801.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model '{EMBED_MODEL}' siap | dimensi: {embedder.get_sentence_embedding_dimension()}")


In [2]:
# ── 1) Embed via sentence-transformers (lokal, no API) ────────────────

def embed_document(text: str) -> list:
    return embedder.encode(text, normalize_embeddings=True).tolist()

def embed_query(text: str) -> list:
    return embedder.encode(text, normalize_embeddings=True, prompt_name="query").tolist()

_dim = len(embed_document("test"))
print(f"Model: {EMBED_MODEL} | dimensi: {_dim}")

Model: BAAI/bge-m3 | dimensi: 1024


In [3]:
# ── 2) Pecah CV JSON jadi chunk + metadata ────────────────────────────

def cv_to_chunks(cv: dict, cv_id: str) -> list:
    chunks = []
    name = cv.get("personal_info", {}).get("name", "")

    def add(kind, text, **meta):
        text = (text or "").strip()
        if text:
            chunks.append({
                "id":   f"{cv_id}::{kind}::{len(chunks)}",
                "text": text,
                "meta": {"cv_id": cv_id, "name": name, "type": kind, **meta},
            })

    add("summary", cv.get("summary", ""))

    sk = cv.get("skills", {})
    all_skills = sk.get("hard_skills", []) + sk.get("soft_skills", [])
    add("skills", "Skills: " + ", ".join(all_skills), skills=", ".join(all_skills))

    for e in cv.get("experience", []):
        ks  = ", ".join(e.get("key_skills", [])) if isinstance(e, dict) else ""
        txt = f'{e.get("role","") if isinstance(e, dict) else ""} at {e.get("company","") if isinstance(e, dict) else ""}. {e.get("summary","") if isinstance(e, dict) else str(e)} Skills: {ks}'
        if isinstance(e, dict):
            add("experience", txt,
                company=e.get("company", ""), role=e.get("role", ""),
                start_date=e.get("start_date", ""), end_date=e.get("end_date", ""),
                is_current=bool(e.get("is_current", False)), key_skills=ks)

    for ed in cv.get("education", []):
        if isinstance(ed, dict):
            txt = f'{ed.get("degree","")} in {ed.get("field_of_study","")} at {ed.get("institution","")}'
            add("education", txt, institution=ed.get("institution", ""))
        else:
            add("education", str(ed), institution="")

    for c in cv.get("certifications", []):
        if isinstance(c, dict):
            add("certification", f'{c.get("name","")} - {c.get("issuer","")}', issuer=c.get("issuer", ""))
        else:
            add("certification", str(c), issuer="")   # handle jika string

    if cv.get("achievements"):
        add("achievements", " ".join(cv["achievements"]))

    return chunks

In [4]:
# ── 3) Build chunks -> embed -> simpan ke ChromaDB ────────────────────
import time

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass
col = chroma_client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

all_chunks = []
files = sorted(PROCESSED_DIR.glob("*.json"))
for jf in files:
    cv = json.load(open(jf, encoding="utf-8"))
    all_chunks += cv_to_chunks(cv, jf.stem)

print(f"{len(all_chunks)} chunk dari {len(files)} CV. Meng-embed...")

embeddings = []
for i, c in enumerate(all_chunks):
    embeddings.append(embed_document(c["text"]))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(all_chunks)} selesai")
        time.sleep(1)

col.add(
    ids=[c["id"]   for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    embeddings=embeddings,
    metadatas=[c["meta"] for c in all_chunks],
)
print(f"Selesai. {col.count()} vektor tersimpan di koleksi '{COLLECTION}'")

58 chunk dari 11 CV. Meng-embed...
  10/58 selesai
  20/58 selesai
  30/58 selesai
  40/58 selesai
  50/58 selesai
Selesai. 58 vektor tersimpan di koleksi 'cv_chunks_mimo'


In [5]:
# ── 4) Preview search ─────────────────────────────────────────────────

def search(requirement: str, n: int = 5, where: dict = None):
    qe  = embed_query(requirement)
    res = col.query(query_embeddings=[qe], n_results=n, where=where)
    print(f"QUERY: {requirement}\n")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f'[{1 - dist:.3f}] {meta.get("name","?"):30} | {meta["type"]:13} | {doc[:80]}')

search("butuh pengalaman backend Java Spring Boot dan microservices")

QUERY: butuh pengalaman backend Java Spring Boot dan microservices

[0.610] Lai Kah Xing (Wilson)          | skills        | Skills: Java, Spring MVC, Spring Boot, Spring Cloud, CI/CD, Jenkins, Kubernetes,
[0.598] RIZA ALAMSYA                   | skills        | Skills: Java, Spring Boot, Java EE, PHP, Laravel, Microservices Architecture, Ev
[0.587] RIZA ALAMSYA                   | experience    | at Agoh Marketing Sdn. Bhd.  Skills: Spring Boot, Microservices Architecture, Ev
[0.566] RIZA ALAMSYA                   | experience    | at PROSPERO.  Skills: Microservices Architecture, Domain-Driven Design (DDD), RE
[0.564] RIZA ALAMSYA                   | experience    | at Collega by Telkom Indonesia.  Skills: Microservices Architecture, Java, Sprin
